<a href="https://colab.research.google.com/github/reza-abdolee/Generative-AI-Systems---Winter-2027/blob/main/Module_08_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 08 Lab: Fine-Tuning Large Language Models (LLMs)
**COMP 490 Generative AI Systems · CSU Channel Islands · Dr. Reza Abdolee**

**Before you start:**
1. Choose **Runtime → Change runtime type → T4 GPU**, then Save.
2. Run the cells **from top to bottom, in order**. The full lab takes about 20–30 minutes on a free T4.
3. This lab does **not** use the NRP API or any API key: the model runs on your Colab GPU.

If you see a CUDA out-of-memory error, choose **Runtime → Restart session** and run the notebook again from the top.

## Part 1 – Environment, GPU Setup, and Model Loading
### Install (takes 2–4 minutes)

In [ ]:
!pip install -q unsloth   # installs matching versions of trl, peft, bitsandbytes, etc.

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                     capture_output=True, text=True).stdout)

### Load the 4-bit model and attach LoRA adapters

In [ ]:
from unsloth import FastLanguageModel   # import unsloth before trl/transformers
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",   # about 30% less GPU memory
    random_state=3407,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable LoRA parameters: {trainable:,} of {total:,} ({100 * trainable / total:.2f}%)")

## Part 2 – Data Preparation

In [ ]:
from datasets import load_dataset

dataset = load_dataset("yahma/alpaca-cleaned", split="train")
print(f"Total training examples loaded: {len(dataset)}")
print(dataset[0])

## Part 3 – Prompt Formatting and Supervised Fine-Tuning
### Step 1: Format every example with the Alpaca template
This adds a `text` field to each example; the trainer reads that field.

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input
that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token   # marks the end of each answer so the model learns to stop

def formatting_prompts_func(examples):
    texts = [alpaca_prompt.format(instruction, inp, output) + EOS_TOKEN
             for instruction, inp, output in zip(examples["instruction"],
                                                 examples["input"], examples["output"])]
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched=True)   # adds the "text" field

print(dataset[0]["text"])

### Step 2: Train for 60 steps (about 10–15 minutes)
Watch the printed loss values.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=1024,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        logging_steps=1,
        output_dir="outputs",
        optim="adamw_8bit",
        seed=3407,
    ),
)

trainer.train()

## Part 4 – Inference with the Fine-Tuned Adapter

In [ ]:
FastLanguageModel.for_inference(model)

test_instruction = "What are the main differences between a Python list and a dictionary?"
formatted_prompt = alpaca_prompt.format(test_instruction, "", "")
inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print(response.split("### Response:")[-1].strip())   # show only the answer

## Part 5 – Qualitative Evaluation: Base Model vs. Fine-Tuned Model
`generate()` runs each prompt twice: once with the LoRA adapter turned off (the base model) and once with it on. The results are saved to `comparison_table.csv` for your report.

In [ ]:
def generate(instruction, use_adapter=True, max_new_tokens=128):
    """Generate an answer with the fine-tuned adapter on, or off (base model)."""
    prompt = alpaca_prompt.format(instruction, "", "")
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    if use_adapter:
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True)
    else:
        with model.disable_adapter():
            outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True)
    text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    return text.split("### Response:")[-1].strip()

import pandas as pd

comparison_prompts = [
    "Explain machine learning to a five-year-old.",
    "Write a short, funny poem about a cat who thinks it is a dog.",
    "If it takes 5 machines 5 minutes to make 5 widgets, how long would it take 100 machines to make 100 widgets?",
]
rows = []
for instruction in comparison_prompts:
    base = generate(instruction, use_adapter=False)
    tuned = generate(instruction, use_adapter=True)
    rows.append({"Prompt": instruction, "Base Model (Before)": base, "Fine-Tuned Model (After)": tuned})
    print(f"===== {instruction} =====\n--- BASE ---\n{base}\n--- FINE-TUNED ---\n{tuned}\n")

comparison_df = pd.DataFrame(rows)
comparison_df.to_csv("comparison_table.csv", index=False)
print("Saved comparison_table.csv")

## Part 6 – Training Loss Visualization and Metrics

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

log_history = trainer.state.log_history
loss_data = [
    {"Step": log["step"], "Training Loss": log["loss"]}
    for log in log_history if "loss" in log
]
loss_df = pd.DataFrame(loss_data)
display(loss_df)

plt.plot(loss_df["Step"], loss_df["Training Loss"], marker=".")
plt.title("Model Training Loss Over 60 Steps")
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.show()

peak_gb = torch.cuda.max_memory_reserved() / 1024**3
total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"Peak GPU memory: {peak_gb:.2f} GB of {total_gb:.2f} GB")

loss_df.to_csv("training_loss.csv", index=False)
print(f"Final training loss: {loss_df['Training Loss'].iloc[-1]:.4f}")

## Optional – Try a smaller model
If your T4 session keeps disconnecting, restart the runtime and change `model_name` in Part 1 to `"unsloth/Llama-3.2-3B-bnb-4bit"`. Everything else stays the same, and training is much faster. Note the change in your report.

## Before you submit
- [ ] Training completed 60 steps (or you explain why not).
- [ ] `comparison_table.csv` and `training_loss.csv` are saved; the loss chart is in your report.
- [ ] Peak GPU memory is recorded.
- [ ] Submit this notebook (Share link or .ipynb) with your Technical Report.